# Master Data Pull: Enrich with Per-Team Batting Stats

This notebook pulls pitch-level Statcast data for **all** games in `master_data.csv` and aggregates
per-team (home vs away) batting statistics. The enriched master is saved back to `master_data.csv`.

### New columns added (16)
| Home Team Batting | Away Team Batting |
|---|---|
| home_bat_hr | away_bat_hr |
| home_bat_k | away_bat_k |
| home_bat_bb | away_bat_bb |
| home_bat_h | away_bat_h |
| home_bat_pitches | away_bat_pitches |
| home_bat_exit_velo | away_bat_exit_velo |
| home_bat_bbe | away_bat_bbe |
| home_bat_hr_h_ratio | away_bat_hr_h_ratio |

### Caching
Statcast data is fetched in 2-week chunks and cached to `statcast_cache/*.parquet`.
Re-running skips already-cached chunks.

**Expected runtime**: ~2-4 hours for full pull (2015-2025). Cached re-runs take minutes.

In [ ]:
import pandas as pd
import numpy as np
from datetime import timedelta
import os
import time
import warnings
warnings.filterwarnings('ignore')

from pybaseball import statcast

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 220)

# --- Load existing master data ---
master = pd.read_csv('master_data.csv')
master['game_date'] = pd.to_datetime(master['game_date'])

print(f"Master dataset: {len(master)} games")
print(f"Date range: {master['game_date'].min().date()} to {master['game_date'].max().date()}")
print(f"Seasons: {sorted(master['season'].unique())}")
print(f"Unique game_pks: {master['game_pk'].nunique()}")

# Check if already enriched
if 'home_bat_hr' in master.columns:
    print("\nWARNING: master_data.csv already has per-team batting columns.")
    print("Re-running will overwrite them.")

## Section 1: Fetch Statcast Data (Cached)

Pull pitch-level data in 2-week chunks. Each chunk is cached as a parquet file
so re-runs skip already-downloaded date ranges.

In [ ]:
CACHE_DIR = 'statcast_cache'
os.makedirs(CACHE_DIR, exist_ok=True)

# All game_pks we need
all_game_pks = set(master['game_pk'].unique())

# Date range for Statcast pulls (season boundaries)
start_date = pd.Timestamp(f"{master['season'].min()}-03-01")
end_date = pd.Timestamp(f"{master['season'].max()}-11-30")

print(f"Statcast fetch range: {start_date.date()} to {end_date.date()}")
print(f"Game PKs to match: {len(all_game_pks)}")
print(f"Cache dir: {os.path.abspath(CACHE_DIR)}")

# Count existing cache files
existing_cache = [f for f in os.listdir(CACHE_DIR) if f.endswith('.parquet')]
print(f"Existing cached chunks: {len(existing_cache)}")

In [ ]:
import signal
import functools

class TimeoutError(Exception):
    pass

def timeout(seconds):
    """Decorator that raises TimeoutError if a function takes too long."""
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            def handler(signum, frame):
                raise TimeoutError(f"Timed out after {seconds} seconds")
            old_handler = signal.signal(signal.SIGALRM, handler)
            signal.alarm(seconds)
            try:
                result = func(*args, **kwargs)
            finally:
                signal.alarm(0)
                signal.signal(signal.SIGALRM, old_handler)
            return result
        return wrapper
    return decorator


@timeout(300)  # 5-minute timeout per chunk
def fetch_one_chunk(start_dt, end_dt):
    """Fetch a single Statcast chunk with a 5-minute timeout."""
    return statcast(start_dt=start_dt, end_dt=end_dt)


def fetch_statcast_all(start_date, end_date, game_pks, chunk_days=14):
    """
    Fetch Statcast data in 2-week chunks, caching each as parquet.
    Prints progress for EVERY chunk. Times out after 5 min per chunk.
    Returns a single DataFrame of all pitch-level data for matching game_pks.
    """
    all_chunks = []
    current = pd.Timestamp(start_date)
    end = pd.Timestamp(end_date)
    total_chunks = int(np.ceil((end - current).days / chunk_days))
    chunk_num = 0
    matched_total = 0

    while current <= end:
        chunk_end = min(current + timedelta(days=chunk_days - 1), end)
        chunk_num += 1
        cache_file = os.path.join(
            CACHE_DIR,
            f"statcast_{current.strftime('%Y%m%d')}_{chunk_end.strftime('%Y%m%d')}.parquet"
        )

        if os.path.exists(cache_file):
            chunk_df = pd.read_parquet(cache_file)
            status = 'cached'
        else:
            t0 = time.time()
            try:
                chunk_df = fetch_one_chunk(
                    start_dt=current.strftime('%Y-%m-%d'),
                    end_dt=chunk_end.strftime('%Y-%m-%d')
                )
                elapsed = time.time() - t0
                if chunk_df is not None and len(chunk_df) > 0:
                    chunk_df.to_parquet(cache_file, index=False)
                    status = f'fetched {len(chunk_df):,} pitches in {elapsed:.0f}s'
                else:
                    chunk_df = pd.DataFrame()
                    status = f'empty ({elapsed:.0f}s)'
            except TimeoutError:
                print(f"  TIMEOUT: {current.date()} to {chunk_end.date()} - skipping (will retry next run)")
                chunk_df = pd.DataFrame()
                status = 'TIMEOUT'
            except Exception as e:
                print(f"  FAILED: {current.date()} to {chunk_end.date()}: {e}")
                chunk_df = pd.DataFrame()
                status = 'FAILED'
            time.sleep(1)  # Be polite to the API

        # Filter to our game_pks
        n_matched = 0
        if len(chunk_df) > 0 and 'game_pk' in chunk_df.columns:
            filtered = chunk_df[chunk_df['game_pk'].isin(game_pks)]
            if len(filtered) > 0:
                all_chunks.append(filtered)
                n_matched = len(filtered)
                matched_total += n_matched

        # Print EVERY chunk
        print(f"  [{chunk_num:3d}/{total_chunks}] {current.date()} to {chunk_end.date()}: "
              f"{status} | {n_matched:,} matched | total matched: {matched_total:,}")

        current = chunk_end + timedelta(days=1)

    print(f"\nDone: {chunk_num} chunks processed")

    if all_chunks:
        result = pd.concat(all_chunks, ignore_index=True)
        print(f"Total matched pitches: {len(result):,}")
        print(f"Unique game_pks with data: {result['game_pk'].nunique():,}")
        return result
    return pd.DataFrame()


# --- Run the fetch ---
print("Fetching Statcast data (this may take a while on first run)...")
print("Each chunk has a 5-minute timeout. Progress printed per chunk.\n")
sc_all = fetch_statcast_all(start_date, end_date, all_game_pks)
print(f"\nStatcast DataFrame shape: {sc_all.shape}")

## Section 2: Aggregate Per-Team Batting Stats

Separate by `inning_topbot`:
- **Bot** (bottom of inning) = **home team batting**
- **Top** (top of inning) = **away team batting**

Aggregate per game per side: HR, K, BB, H, pitches, exit_velo, barrels, BBE, barrel_rate, hr_h_ratio.

In [ ]:
def aggregate_batting_stats(sc_df):
    """
    Aggregate pitch-level Statcast data to per-game, per-side batting stats.
    
    Returns DataFrame with game_pk + 16 batting columns (8 home, 8 away).
    Barrel stats excluded (not reliably available in all pybaseball versions).
    """
    if len(sc_df) == 0:
        return pd.DataFrame()

    def agg_side(group):
        events = group['events'].dropna()
        return pd.Series({
            'bat_hr': (events == 'home_run').sum(),
            'bat_k': events.str.contains('strikeout', na=False).sum(),
            'bat_bb': (events == 'walk').sum(),
            'bat_h': events.isin(['single', 'double', 'triple', 'home_run']).sum(),
            'bat_pitches': len(group),
            'bat_exit_velo': group['launch_speed'].dropna().mean() if group['launch_speed'].notna().any() else np.nan,
            'bat_bbe': group['launch_speed'].notna().sum(),
        })

    results = []
    game_pks = sc_df['game_pk'].unique()
    total = len(game_pks)

    for i, game_pk in enumerate(game_pks):
        game_df = sc_df[sc_df['game_pk'] == game_pk]
        row = {'game_pk': game_pk}

        for side, topbot in [('home', 'Bot'), ('away', 'Top')]:
            side_df = game_df[game_df['inning_topbot'] == topbot]
            if len(side_df) > 0:
                stats = agg_side(side_df)
            else:
                stats = pd.Series({
                    'bat_hr': 0, 'bat_k': 0, 'bat_bb': 0, 'bat_h': 0,
                    'bat_pitches': 0, 'bat_exit_velo': np.nan,
                    'bat_bbe': 0,
                })

            for stat_name, val in stats.items():
                row[f'{side}_{stat_name}'] = val

        # Derived stats
        for side in ['home', 'away']:
            h = row[f'{side}_bat_h']
            hr = row[f'{side}_bat_hr']
            row[f'{side}_bat_hr_h_ratio'] = hr / h if h > 0 else np.nan

        results.append(row)

        if (i + 1) % 5000 == 0:
            print(f"  Aggregated {i + 1}/{total} games...")

    return pd.DataFrame(results)


print("Aggregating per-team batting stats...")
batting_df = aggregate_batting_stats(sc_all)
print(f"\nBatting stats: {len(batting_df)} games x {len(batting_df.columns)} columns")
print(f"\nSample:")
batting_df.head(3)

## Section 3: Merge & Validate

Merge per-team batting stats into master_data and run validation checks.

In [ ]:
# --- Merge batting stats into master ---
# Drop existing batting columns if re-running
bat_cols = [c for c in master.columns if c.startswith('home_bat_') or c.startswith('away_bat_')]
if bat_cols:
    print(f"Dropping {len(bat_cols)} existing batting columns before merge")
    master = master.drop(columns=bat_cols)

enriched = master.merge(batting_df, on='game_pk', how='left')

print(f"Master before: {master.shape}")
print(f"Master after:  {enriched.shape}")
print(f"\nNew columns: {[c for c in enriched.columns if c not in master.columns]}")

# --- Check coverage ---
n_with_batting = enriched['home_bat_hr'].notna().sum()
n_missing = enriched['home_bat_hr'].isna().sum()
print(f"\nGames with batting data: {n_with_batting} ({100*n_with_batting/len(enriched):.1f}%)")
print(f"Games missing batting data: {n_missing} ({100*n_missing/len(enriched):.1f}%)")

In [ ]:
# --- Validation ---
print("=" * 70)
print("VALIDATION")
print("=" * 70)

# 1. Strikeouts check: home_bat_k + away_bat_k ≈ strikeouts
valid = enriched.dropna(subset=['home_bat_k', 'away_bat_k'])
valid = valid.copy()
valid['k_sum'] = valid['home_bat_k'] + valid['away_bat_k']
valid['k_diff'] = abs(valid['k_sum'] - valid['strikeouts'])
pct_k_close = (valid['k_diff'] <= 2).mean() * 100
print(f"\n1. Strikeouts: home_bat_k + away_bat_k within ±2 of combined: {pct_k_close:.1f}%")
print(f"   Mean diff: {valid['k_diff'].mean():.2f}, Max diff: {valid['k_diff'].max():.0f}")

# 2. Home runs check: home_bat_hr + away_bat_hr ≈ home_runs_hit
valid['hr_sum'] = valid['home_bat_hr'] + valid['away_bat_hr']
valid['hr_diff'] = abs(valid['hr_sum'] - valid['home_runs_hit'])
pct_hr_close = (valid['hr_diff'] <= 1).mean() * 100
print(f"\n2. Home runs: home_bat_hr + away_bat_hr within ±1 of combined: {pct_hr_close:.1f}%")
print(f"   Mean diff: {valid['hr_diff'].mean():.2f}, Max diff: {valid['hr_diff'].max():.0f}")

# 3. Null rates in new columns
print(f"\n3. Null rates in per-team batting columns:")
new_cols = [c for c in enriched.columns if c.startswith('home_bat_') or c.startswith('away_bat_')]
for col in new_cols:
    null_pct = enriched[col].isna().mean() * 100
    status = 'OK' if null_pct < 5 else 'WARNING'
    print(f"   {col}: {null_pct:.1f}% null [{status}]")

# 4. Summary stats for new columns
print(f"\n4. Summary stats:")
print(enriched[new_cols].describe().round(3).T[['mean', 'std', 'min', 'max']].to_string())

## Section 4: Save Enriched Master Data

In [ ]:
# Round floating point columns
round_map = {
    'home_bat_exit_velo': 1, 'away_bat_exit_velo': 1,
    'home_bat_hr_h_ratio': 4, 'away_bat_hr_h_ratio': 4,
}
for col, decimals in round_map.items():
    if col in enriched.columns:
        enriched[col] = enriched[col].round(decimals)

# Save
enriched.to_csv('master_data.csv', index=False)

print(f"Saved enriched master_data.csv")
print(f"  Rows: {len(enriched)}, Columns: {len(enriched.columns)}")
print(f"  File size: {os.path.getsize('master_data.csv') / 1024:.1f} KB")

# Verify roundtrip
verify = pd.read_csv('master_data.csv')
assert verify.shape == enriched.shape, f"Shape mismatch: {verify.shape} vs {enriched.shape}"
print(f"\nRoundtrip verification: PASSED")
print(f"\nColumn list ({len(enriched.columns)}):")
for i, col in enumerate(enriched.columns):
    print(f"  {i+1:2d}. {col}")